# Exp8 & Exp9: anion deficiency, oxygen, and composition-space neighborhoods

Read-only exploratory analysis of visible `DTUCombinatorialSample` records. Exp8 tests whether measured oxygen is associated with a deficit in a user-selected chalcogen relative to an explicitly entered phase stoichiometry. Exp9 overlays user-suggested phase compositions on ternary sample maps and shows nearest-composition regions as a geometric visualization.

**Interpretation limits:** a composition-nearest region is not a thermodynamic or kinetic basin of attraction. CIF compositions are treated as user-designated stable references, but this notebook does not independently verify hull stability or calculate formation energies, substitution energetics, or Vladan's theory. Aimed/unplanned labels are user-reported. Multiple grid samples from one library/process are correlated observations, not independent experiments. Samples missing any element required by their material-space/reference formula are excluded rather than silently treating that element as zero.

The same notebook runs locally and in NOMAD NORTH. It queries only the fields needed and does not write to NOMAD. Records are limited by the permissions of the authenticated account (`owner: visible`).

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'notebooks' / 'oasis_trends'):
    if (candidate / 'oasis_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not find notebooks/oasis_trends/oasis_helpers.py; start Jupyter in the repository or notebook folder.')

import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from oasis_helpers import query_archives

SAMPLE_ENTRY_TYPE = 'DTUCombinatorialSample'
MAX_SAMPLES = 100_000       # current visible count observed: about 23k
QUERY_PAGE_SIZE = 200
INCLUDE_SUBSYSTEMS = True   # include binary samples on relevant ternary edges in Exp9
PLOTLY_CONFIG = {'toImageButtonOptions': {'format': 'svg', 'scale': 1}}
PAULING_EN = {
    'Ag': 1.93, 'Ba': 0.89, 'Cu': 1.90, 'N': 3.04, 'P': 2.19, 'S': 2.58,
    'Sb': 2.05, 'Se': 2.55, 'Sn': 1.96, 'Te': 2.10, 'Zn': 1.65, 'Zr': 1.33,
}

records, server_total = query_archives(
    query={'entry_type:all': [SAMPLE_ENTRY_TYPE]},
    required_data={
        'lab_id': '*',
        'deposition': {'material_space': '*'},
        'elemental_composition': '*',
    },
    limit=MAX_SAMPLES,
    page_size=QUERY_PAGE_SIZE,
)

rows = []
for entry_id, data in records:
    deposition = data.get('deposition') or {}
    composition = {}
    for item in data.get('elemental_composition') or []:
        if isinstance(item, dict) and item.get('element') and item.get('atomic_fraction') is not None:
            composition[item['element']] = float(item['atomic_fraction'])
    rows.append({
        'entry_id': entry_id,
        'lab_id': data.get('lab_id'),
        'material_space': deposition.get('material_space'),
        'composition': composition,
    })
samples = pd.DataFrame(rows)
del records, rows
print(f'Fetched {len(samples):,} partial archives; server reports {server_total!r} visible samples; limit={MAX_SAMPLES:,}.')

In [ ]:
def elements_in(space):
    if not isinstance(space, str):
        return tuple()
    tokens = [x for x in re.split(r'[\s,;\-_/]+', space.strip()) if x]
    return tuple(sorted(set(tokens), key=lambda x: (PAULING_EN.get(x, float('inf')), x)))


samples['elements'] = samples['material_space'].map(elements_in)
samples['space_key'] = samples['elements'].map(lambda x: '-'.join(x))
def inferred_lab_id_family(value):
    # Many grid-sample IDs end in a numeric position; this prefix is a
    # sensitivity grouping only, not a guaranteed NOMAD library identifier.
    return re.sub(r'_\d+$', '', value) if isinstance(value, str) else None
samples['lab_id_family'] = samples['lab_id'].map(inferred_lab_id_family)
def normalize_reported_composition(composition):
    total = sum(composition.values())
    return {element: value / total for element, value in composition.items()} if total > 0 else {}
samples['normalized_composition'] = samples['composition'].map(normalize_reported_composition)
samples['oxygen_at_fraction'] = samples['normalized_composition'].map(lambda c: c.get('O', np.nan))
summary = pd.DataFrame([
    {'measure': 'records returned', 'count': len(samples)},
    {'measure': 'records with material_space', 'count': int(samples['material_space'].notna().sum())},
    {'measure': 'records with composition', 'count': int(samples['composition'].map(bool).sum())},
    {'measure': 'records with both', 'count': int((samples['material_space'].notna() & samples['composition'].map(bool)).sum())},
    {'measure': 'records with reported oxygen', 'count': int(samples['oxygen_at_fraction'].notna().sum())},
])
samples['reported_fraction_sum'] = samples['composition'].map(lambda c: sum(c.values()) if c else np.nan)
display(summary)
print('Elemental-composition fraction-sum quantiles (quality check):')
display(samples['reported_fraction_sum'].describe(percentiles=[0.1, 0.5, 0.9]).to_frame())
space_counts = samples.groupby('space_key').size().sort_values(ascending=False).rename('n_samples').to_frame()
display(space_counts.head(30))

In [ ]:
EXP9_PHASES = [
    {'label': 'Ag2PS3', 'system': 'Ag-P-S', 'stoichiometry': {'Ag': 2, 'P': 1, 'S': 3}, 'intent': 'Intent not supplied', 'source': 'Ag2PS3_v1_o1.cif'},
    {'label': 'Ag4P2S7', 'system': 'Ag-P-S', 'stoichiometry': {'Ag': 4, 'P': 2, 'S': 7}, 'intent': 'Intent not supplied', 'source': 'Ag4P2S7_o1.cif'},
    {'label': 'Ag7PS6', 'system': 'Ag-P-S', 'stoichiometry': {'Ag': 7, 'P': 1, 'S': 6}, 'intent': 'Intent not supplied', 'source': 'Ag7PS6_v1_o1.cif'},
    {'label': 'AgPS3', 'system': 'Ag-P-S', 'stoichiometry': {'Ag': 1, 'P': 1, 'S': 3}, 'intent': 'Intent not supplied', 'source': 'AgPS3_v1_o1.cif'},
    {'label': 'BaTe2P4', 'system': 'Ba-P-Te', 'stoichiometry': {'Ba': 1, 'P': 4, 'Te': 2}, 'intent': 'Intent not supplied', 'source': 'Ba(TeP2)2.cif'},
    {'label': 'Ba3P2S8', 'system': 'Ba-P-S', 'stoichiometry': {'Ba': 3, 'P': 2, 'S': 8}, 'intent': 'Intent not supplied', 'source': 'Ba3P2S8_v1_o2.cif'},
    {'label': 'Ba4P2S', 'system': 'Ba-P-S', 'stoichiometry': {'Ba': 4, 'P': 2, 'S': 1}, 'intent': 'Intent not supplied', 'source': 'Ba4P2S_v1_o2.cif'},
    {'label': 'BaPS3', 'system': 'Ba-P-S', 'stoichiometry': {'Ba': 1, 'P': 1, 'S': 3}, 'intent': 'Intent not supplied', 'source': 'BaPS3_v1_o2.cif'},
    {'label': 'Cu3PS4', 'system': 'Cu-P-S', 'stoichiometry': {'Cu': 3, 'P': 1, 'S': 4}, 'intent': 'Aimed (user-reported)', 'source': 'Cu3PS4.cif'},
    {'label': 'Cu7PS6', 'system': 'Cu-P-S', 'stoichiometry': {'Cu': 7, 'P': 1, 'S': 6}, 'intent': 'Unplanned (user-reported)', 'source': 'Cu7PS6_v1_o1.cif'},
    {'label': 'Cu3P', 'system': 'Cu-P-S', 'stoichiometry': {'Cu': 3, 'P': 1}, 'intent': 'Unplanned (user-reported)', 'source': 'no CIF supplied'},
    {'label': 'Cu8P2S', 'system': 'Cu-P-S', 'stoichiometry': {'Cu': 8, 'P': 2, 'S': 1}, 'intent': 'Intent not supplied', 'source': 'Cu8P2S_v1_o1.cif'},
    {'label': 'CuPS', 'system': 'Cu-P-S', 'stoichiometry': {'Cu': 1, 'P': 1, 'S': 1}, 'intent': 'Intent not supplied', 'source': 'CuPS.cif'},
    {'label': 'CuPS3', 'system': 'Cu-P-S', 'stoichiometry': {'Cu': 1, 'P': 1, 'S': 3}, 'intent': 'Intent not supplied', 'source': 'CuPS3_v1_o1.cif'},
    {'label': 'Sb2PS2', 'system': 'Sb-P-S', 'stoichiometry': {'Sb': 2, 'P': 1, 'S': 2}, 'intent': 'Intent not supplied', 'source': 'Sb2PS2.cif'},
    {'label': 'Sb3PS6', 'system': 'Sb-P-S', 'stoichiometry': {'Sb': 3, 'P': 1, 'S': 6}, 'intent': 'Intent not supplied', 'source': 'Sb3PS6.cif'},
    {'label': 'SbP', 'system': 'Sb-P-S', 'stoichiometry': {'Sb': 1, 'P': 1}, 'intent': 'Intent not supplied', 'source': 'SbP.cif'},
    {'label': 'SbP2', 'system': 'Sb-P-S', 'stoichiometry': {'Sb': 1, 'P': 2}, 'intent': 'Intent not supplied', 'source': 'SbP2.cif'},
    {'label': 'SnP2S6', 'system': 'Sn-P-S', 'stoichiometry': {'Sn': 1, 'P': 2, 'S': 6}, 'intent': 'Intent not supplied', 'source': 'SnP2S6_o4.cif'},
    {'label': 'SnPS3', 'system': 'Sn-P-S', 'stoichiometry': {'Sn': 1, 'P': 1, 'S': 3}, 'intent': 'Unplanned (user-reported)', 'source': 'SnPS3_HSE.cif; formula from atom-site counts'},
    {'label': 'SnS', 'system': 'Sn-P-S', 'stoichiometry': {'Sn': 1, 'S': 1}, 'intent': 'Unplanned (user-reported)', 'source': 'no CIF supplied'},
    {'label': 'Zr2SN2', 'system': 'Zr-S-N', 'stoichiometry': {'Zr': 2, 'S': 1, 'N': 2}, 'intent': 'Aimed (user-reported)', 'source': 'Zr2SN2-alpha_EntryWithCollCode96970.cif'},
    {'label': 'ZrN', 'system': 'Zr-S-N', 'stoichiometry': {'Zr': 1, 'N': 1}, 'intent': 'Intent not supplied', 'source': 'ZrN.cif'},
    {'label': 'ZrN2', 'system': 'Zr-S-N', 'stoichiometry': {'Zr': 1, 'N': 2}, 'intent': 'Intent not supplied', 'source': 'ZrN2.cif'},
    {'label': 'Zr2SeN2', 'system': 'Zr-Se-N', 'stoichiometry': {'Zr': 2, 'Se': 1, 'N': 2}, 'intent': 'Intent not supplied', 'source': 'user; no CIF supplied'},
    {'label': 'ZrZnN2', 'system': 'Zr-Zn-N', 'stoichiometry': {'Zr': 1, 'Zn': 1, 'N': 2}, 'intent': 'Aimed (user-reported)', 'source': 'ZrZnN2.cif'},
]


## Exp8: oxygen versus anion deficit from candidate phase stoichiometry

Exp8 uses the shared candidate phase list for all systems with a supplied S, Se or Te formula; phases without that chalcogen are listed as unavailable. Formula, target chalcogen and matrix denominator are all explicit: this is not just replacing element names. For binary references in a ternary material space, the matrix is the other element in the formula, even when the third system element is present in the measured sample. A reference is a composition comparison, not a phase assignment. Deficits are clipped at zero, so changing reference can change rank correlations through ties. For each phase, compare O and anion deficiency both as atomic fractions and relative to the named matrix, at point and inferred-family levels, and inspect fraction-sum sensitivity. Samples missing any element in the formula or O are excluded; absent O is never set to zero. Reusing the same samples for multiple references does not create independent evidence.


In [ ]:
# Candidate chalcogen chosen per material space; other anions need a separate hypothesis.
EXP8_ANION_BY_SYSTEM = {
    'Ag-P-S': 'S', 'Ba-P-S': 'S', 'Ba-P-Te': 'Te', 'Cu-P-S': 'S',
    'Sb-P-S': 'S', 'Sn-P-S': 'S', 'Zr-S-N': 'S', 'Zr-Se-N': 'Se',
}
EXP8_REFERENCES = []
reference_coverage = []
for phase_ref in EXP9_PHASES:
    anion = EXP8_ANION_BY_SYSTEM.get(phase_ref['system'])
    counts = phase_ref['stoichiometry']
    if anion is None or anion not in counts:
        reference_coverage.append({'system': phase_ref['system'], 'phase': phase_ref['label'],
                                   'status': 'No selected chalcogen in formula/system'})
        continue
    matrix = [el for el in counts if el != anion]
    EXP8_REFERENCES.append({'system': phase_ref['system'], 'phase': phase_ref['label'],
                            'stoichiometry': counts, 'anion': anion,
                            'matrix_elements': matrix, 'status': 'user-supplied candidate',
                            'source': phase_ref['source']})
    reference_coverage.append({'system': phase_ref['system'], 'phase': phase_ref['label'],
                               'status': 'Selected', 'anion': anion, 'matrix': '+'.join(matrix)})
print('Reference selection and matrix denominators:')
display(pd.DataFrame(reference_coverage))
available_keys = {'-'.join(elements_in(ref['system'])) for ref in EXP8_REFERENCES}
chalcogen_spaces = samples[samples['elements'].map(lambda elements: bool({'S', 'Se', 'Te'} & set(elements)))]
chalcogen_coverage = chalcogen_spaces.groupby('space_key').size().rename('n_samples').reset_index()
chalcogen_coverage['has_candidate_reference'] = chalcogen_coverage['space_key'].isin(available_keys)
print('Visible material spaces containing S, Se or Te, including those without a selected reference:')
display(chalcogen_coverage.sort_values('n_samples', ascending=False))
EXP8_PLOT_PHASE = 'SnPS3'  # Exact reference label for plots; summaries always include all references.

exp8_frames, pending = [], []
for ref in EXP8_REFERENCES:
    if not ref['stoichiometry']:
        pending.append({'system': ref['system'], 'phase': ref['phase'], 'anion': ref['anion'], 'status': ref['status']})
        continue
    counts = ref['stoichiometry']
    if ref['anion'] not in counts or sum(counts.values()) <= 0:
        raise ValueError(f"Reference {ref['phase']} must include positive stoichiometry for {ref['anion']}.")
    expected_anion = counts[ref['anion']] / sum(counts.values())
    matrix_elements = ref['matrix_elements']
    if not matrix_elements or any(el not in counts for el in matrix_elements):
        raise ValueError(f"Reference {ref['phase']} must define its non-target matrix elements.")
    expected_anion_per_matrix = counts[ref['anion']] / sum(counts[el] for el in matrix_elements)
    key = '-'.join(elements_in(ref['system']))
    system_samples = samples.loc[samples['space_key'] == key,
        ['entry_id', 'lab_id', 'lab_id_family', 'normalized_composition',
         'oxygen_at_fraction', 'reported_fraction_sum']].copy()
    complete_formula = system_samples['normalized_composition'].map(lambda c: set(counts).issubset(c))
    chosen = system_samples[complete_formula & system_samples['oxygen_at_fraction'].notna()].copy()
    chosen['measured_anion_at_fraction'] = chosen['normalized_composition'].map(lambda c: c[ref['anion']])
    chosen['matrix_at_fraction'] = chosen['normalized_composition'].map(lambda c: sum(c.get(el, 0.0) for el in matrix_elements))
    denominator = chosen['matrix_at_fraction'].where(chosen['matrix_at_fraction'] > 0)
    chosen['measured_anion_per_matrix'] = chosen['measured_anion_at_fraction'] / denominator
    chosen['oxygen_per_matrix'] = chosen['oxygen_at_fraction'] / denominator
    chosen['reference_anion_per_matrix'] = expected_anion_per_matrix
    chosen['anion_ratio_deficit'] = (
        chosen['reference_anion_per_matrix'] - chosen['measured_anion_per_matrix']
    ).clip(lower=0)
    chosen['reference_anion_at_fraction'] = expected_anion
    chosen['anion_deficit_at_fraction'] = (
        chosen['reference_anion_at_fraction'] - chosen['measured_anion_at_fraction']
    ).clip(lower=0)
    chosen['phase'] = ref['phase']
    chosen['system'] = ref['system']
    chosen['anion'] = ref['anion']
    chosen['matrix_elements'] = '+'.join(matrix_elements)
    chosen['reference_source'] = ref['source']
    chosen['n_system_samples'] = len(system_samples)
    chosen['n_complete_formula_samples'] = int(complete_formula.sum())
    chosen['n_complete_formula_with_oxygen'] = len(chosen)
    if chosen.empty:
        print(f"No eligible samples for {ref['system']} / {ref['phase']} (system samples: {len(system_samples)}).")
    else:
        exp8_frames.append(chosen.drop(columns=['normalized_composition']))

if pending:
    print('Exp8 reference phases awaiting formula confirmation:')
    display(pd.DataFrame(pending))
exp8 = pd.concat(exp8_frames, ignore_index=True) if exp8_frames else pd.DataFrame()
if exp8.empty:
    print('No Exp8 rows with both reported oxygen and a configured reference phase.')
else:
    exp8_summary = exp8.groupby(['system', 'phase', 'anion']).agg(
        n_samples=('entry_id', 'size'),
        matrix=('matrix_elements', 'first'), reference_source=('reference_source', 'first'),
        n_positive_deficit=('anion_deficit_at_fraction', lambda x: int((x > 0).sum())),
        n_positive_ratio_deficit=('anion_ratio_deficit', lambda x: int((x > 0).sum())),
        n_system_samples=('n_system_samples', 'first'),
        n_complete_formula_samples=('n_complete_formula_samples', 'first'),
        n_complete_formula_with_oxygen=('n_complete_formula_with_oxygen', 'first'),
        n_lab_ids=('lab_id', 'nunique'),
        n_inferred_lab_id_families=('lab_id_family', 'nunique'),
        oxygen_median=('oxygen_at_fraction', 'median'),
        oxygen_mean=('oxygen_at_fraction', 'mean'),
        anion_deficit_median=('anion_deficit_at_fraction', 'median'),
    ).reset_index()
    sample_rho = pd.DataFrame([
        {'system': system, 'phase': phase,
         'sample_level_spearman_rho': g['oxygen_at_fraction'].corr(g['anion_deficit_at_fraction'], method='spearman'),
         'sample_ratio_spearman_rho': g['oxygen_per_matrix'].corr(g['anion_ratio_deficit'], method='spearman')}
        for (system, phase), g in exp8.groupby(['system', 'phase'])
    ])
    exp8_summary = exp8_summary.merge(sample_rho, on=['system', 'phase'], how='left')
    strict = exp8[exp8['reported_fraction_sum'].between(0.9, 1.1)]
    strict_rho = pd.DataFrame([
        {'system': system, 'phase': phase,
         'rho_fraction_sum_0_9_to_1_1': g['oxygen_at_fraction'].corr(g['anion_deficit_at_fraction'], method='spearman'),
         'ratio_rho_fraction_sum_0_9_to_1_1': g['oxygen_per_matrix'].corr(g['anion_ratio_deficit'], method='spearman'),
         'n_fraction_sum_0_9_to_1_1': len(g)}
        for (system, phase), g in strict.groupby(['system', 'phase'])
    ])
    exp8_summary = exp8_summary.merge(strict_rho, on=['system', 'phase'], how='left')
    display(exp8_summary)
    family_data = exp8.dropna(subset=['lab_id_family']).groupby(
        ['system', 'phase', 'lab_id_family'], as_index=False
    ).agg(
        n_spatial_samples=('entry_id', 'size'),
        oxygen_at_fraction=('oxygen_at_fraction', 'mean'),
        anion_deficit_at_fraction=('anion_deficit_at_fraction', 'mean'),
        oxygen_per_matrix=('oxygen_per_matrix', 'mean'),
        anion_ratio_deficit=('anion_ratio_deficit', 'mean'),
    )
    family_rho = pd.DataFrame([
        {'system': system, 'phase': phase,
         'inferred_family_spearman_rho': g['oxygen_at_fraction'].corr(g['anion_deficit_at_fraction'], method='spearman'),
         'inferred_family_ratio_rho': g['oxygen_per_matrix'].corr(g['anion_ratio_deficit'], method='spearman')}
        for (system, phase), g in family_data.groupby(['system', 'phase'])
    ])
    def family_bootstrap_interval(group, x, y, n_bootstraps=1000, seed=42):
        paired = group[[x, y]].dropna().reset_index(drop=True)
        if len(paired) < 6 or paired[x].nunique() < 2 or paired[y].nunique() < 2:
            return (np.nan, np.nan, 0)
        rng = np.random.default_rng(seed)
        estimates = []
        for _ in range(n_bootstraps):
            sample = paired.iloc[rng.integers(0, len(paired), size=len(paired))]
            if sample[x].nunique() > 1 and sample[y].nunique() > 1:
                estimates.append(sample[x].corr(sample[y], method='spearman'))
        if len(estimates) < 0.9 * n_bootstraps:
            return (np.nan, np.nan, len(estimates))
        return (*np.percentile(estimates, [2.5, 97.5]), len(estimates))
    family_bootstrap_rows = []
    for (system, phase), group in family_data.groupby(['system', 'phase']):
        for suffix, x_col, y_col in [
            ('fraction', 'anion_deficit_at_fraction', 'oxygen_at_fraction'),
            ('matrix_ratio', 'anion_ratio_deficit', 'oxygen_per_matrix'),
        ]:
            low, high, n_valid = family_bootstrap_interval(group, x_col, y_col)
            family_bootstrap_rows.append({'system': system, 'phase': phase, 'metric': suffix,
                                          'ci_2_5': low, 'ci_97_5': high,
                                          'valid_bootstraps': n_valid})
    family_bootstrap_summary = pd.DataFrame(family_bootstrap_rows)
    print('Second-round sensitivity: aggregate by inferred lab-ID family; also exclude composition totals outside 0.9-1.1.')
    display(family_data.groupby(['system', 'phase']).agg(
        n_inferred_families=('lab_id_family', 'nunique'),
        median_family_O=('oxygen_at_fraction', 'median'),
        median_family_anion_deficit=('anion_deficit_at_fraction', 'median'),
    ).reset_index().merge(family_rho, on=['system', 'phase'], how='left'))
    strict_family = strict.dropna(subset=['lab_id_family']).groupby(
        ['system', 'phase', 'lab_id_family'], as_index=False
    ).agg(
        oxygen_at_fraction=('oxygen_at_fraction', 'mean'),
        anion_deficit_at_fraction=('anion_deficit_at_fraction', 'mean'),
        oxygen_per_matrix=('oxygen_per_matrix', 'mean'),
        anion_ratio_deficit=('anion_ratio_deficit', 'mean'),
    )
    strict_family_rho = pd.DataFrame([
        {'system': system, 'phase': phase,
         'strict_inferred_family_rho': g['oxygen_at_fraction'].corr(g['anion_deficit_at_fraction'], method='spearman'),
         'strict_inferred_family_ratio_rho': g['oxygen_per_matrix'].corr(g['anion_ratio_deficit'], method='spearman'),
         'n_strict_families': len(g)}
        for (system, phase), g in strict_family.groupby(['system', 'phase'])
    ])
    display(strict_family_rho)
    print('Inferred-family bootstrap intervals (families are inferred from lab-ID suffixes, not verified library replicates):')
    display(family_bootstrap_summary)
    if EXP8_PLOT_PHASE not in exp8['phase'].values:
        print(f'No eligible samples for {EXP8_PLOT_PHASE!r}; plotting the first available reference instead.')
        EXP8_PLOT_PHASE = exp8['phase'].iloc[0]
    plot_family = family_data[family_data['phase'] == EXP8_PLOT_PHASE]
    plot_exp8 = exp8[exp8['phase'] == EXP8_PLOT_PHASE]
    if len(plot_family):
        family_fig = px.scatter(
            plot_family, x='anion_deficit_at_fraction', y='oxygen_at_fraction', color='system',
            hover_data=['lab_id_family', 'n_spatial_samples'],
            labels={'anion_deficit_at_fraction': 'Family-mean anion deficit (atomic fraction)',
                    'oxygen_at_fraction': 'Family-mean O atomic fraction'},
            title='Exp8 sensitivity: one mean per inferred lab-ID family',
            template='plotly_white',
        )
        family_fig.update_xaxes(showline=True, mirror=True, linecolor='black')
        family_fig.update_yaxes(showline=True, mirror=True, linecolor='black')
        family_fig.show(config=PLOTLY_CONFIG)
    ratio_fig = px.scatter(
        plot_family, x='anion_ratio_deficit', y='oxygen_per_matrix', color='system',
        hover_data=['lab_id_family', 'n_spatial_samples'],
        labels={'anion_ratio_deficit': 'Deficit in anion / matrix-atom ratio',
                'oxygen_per_matrix': 'O / matrix-atom ratio'},
        title='Exp8 sensitivity: oxygen and anion deficit normalized to the non-target matrix',
        template='plotly_white',
    )
    ratio_fig.update_xaxes(showline=True, mirror=True, linecolor='black')
    ratio_fig.update_yaxes(showline=True, mirror=True, linecolor='black')
    ratio_fig.show(config=PLOTLY_CONFIG)
    def deficit_quartiles(values):
        n_bins = min(4, values.nunique())
        if n_bins < 2:
            return pd.Series(['single value'] * len(values), index=values.index)
        labels = ['Q1 low', 'Q2', 'Q3', 'Q4 high'][:n_bins]
        return pd.qcut(values.rank(method='first'), n_bins, labels=labels)
    exp8['deficit_quartile'] = exp8.groupby(['system', 'phase'])['anion_deficit_at_fraction'].transform(deficit_quartiles)
    quartile_summary = exp8.groupby(['system', 'phase', 'deficit_quartile'], observed=True).agg(
        n_samples=('entry_id', 'size'),
        median_oxygen=('oxygen_at_fraction', 'median'),
        median_anion_deficit=('anion_deficit_at_fraction', 'median'),
    ).reset_index()
    display(quartile_summary)
    exp8['ratio_deficit_quartile'] = exp8.groupby(['system', 'phase'])['anion_ratio_deficit'].transform(deficit_quartiles)
    ratio_quartile_summary = exp8.groupby(['system', 'phase', 'ratio_deficit_quartile'], observed=True).agg(
        n_samples=('entry_id', 'size'),
        median_oxygen_per_matrix=('oxygen_per_matrix', 'median'),
        median_ratio_deficit=('anion_ratio_deficit', 'median'),
    ).reset_index()
    display(ratio_quartile_summary)
    fig = px.scatter(
        plot_exp8, x='anion_deficit_at_fraction', y='oxygen_at_fraction', color='system',
        hover_data=['lab_id', 'phase', 'measured_anion_at_fraction'], opacity=0.45,
        labels={'anion_deficit_at_fraction': 'Anion deficit vs. reference (atomic fraction)',
                'oxygen_at_fraction': 'Measured O atomic fraction'},
        title='Exp8: measured oxygen versus reference-anion deficit',
        template='plotly_white',
    )
    fig.update_xaxes(showline=True, mirror=True, linecolor='black')
    fig.update_yaxes(showline=True, mirror=True, linecolor='black')
    fig.show(config=PLOTLY_CONFIG)
    px.box(
        exp8[exp8['phase'] == EXP8_PLOT_PHASE], x='deficit_quartile', y='oxygen_at_fraction', color='system', points='all',
        labels={'deficit_quartile': 'Anion-deficit quartile', 'oxygen_at_fraction': 'Measured O atomic fraction'},
        title='Exp8: oxygen across within-system anion-deficit quartiles',
        template='plotly_white',
    ).show(config=PLOTLY_CONFIG)
    px.box(
        exp8[exp8['phase'] == EXP8_PLOT_PHASE], x='ratio_deficit_quartile', y='oxygen_per_matrix', color='system', points='all',
        labels={'ratio_deficit_quartile': 'Anion/matrix-ratio-deficit quartile',
                'oxygen_per_matrix': 'O / matrix-atom ratio'},
        title='Exp8 sensitivity: oxygen across matrix-normalized anion-deficit quartiles',
        template='plotly_white',
    ).show(config=PLOTLY_CONFIG)

## Exp9: measured compositions around candidate phases

The phase list and aimed/unplanned labels below are based on the supplied CIFs and user's examples; CIF compositions are treated as user-designated stable references, not independently checked against an energy convex hull. For each ternary system, the faint background is a nearest-reference Voronoi partition in normalized composition coordinates; it is an illustrative geometric map, **not** a thermodynamic basin. Binary sample spaces and binary candidate phases can lie on ternary edges. Ternary axes are renormalized over the three plotted elements only; oxygen and other elements are excluded from axes and shown in hover where present.

In [ ]:
VLADAN_THEORY_MAPPING = None  # add a specific, user-confirmed prediction before testing it
GRID_DIVISIONS = 30

phase_rows = []
for ref in EXP9_PHASES:
    system = elements_in(ref['system'])
    if len(system) != 3:
        raise ValueError(f"Exp9 system {ref['system']} is not ternary.")
    amount = sum(ref['stoichiometry'].values())
    phase_comp = {el: ref['stoichiometry'].get(el, 0) / amount for el in system}
    phase_rows.append({**ref, 'system_key': '-'.join(system), 'elements': system, 'phase_comp': phase_comp})
phases = pd.DataFrame(phase_rows)
display(phases[['label', 'system_key', 'intent', 'phase_comp']])

palette = px.colors.qualitative.Dark24
figures = {}
for system_key, phase_group in phases.groupby('system_key', sort=False):
    system = tuple(system_key.split('-'))
    system_set = set(system)
    refs = phase_group.to_dict('records')
    ref_by_label = {r['label']: r for r in refs}
    colors = {r['label']: palette[i % len(palette)] for i, r in enumerate(refs)}
    if len(refs) == 1:
        print(f'{system_key}: only one reference formula; nearest-phase labels are automatic, so interpret this as a distance map, not a basin comparison.')

    valid_space = samples['elements'].map(
        lambda els: bool(els) and (set(els) <= system_set if INCLUDE_SUBSYSTEMS else set(els) == system_set)
    )
    subset = samples[valid_space & samples['composition'].map(bool)].copy()
    has_space_elements = subset.apply(lambda row: set(row['elements']).issubset(row['composition']), axis=1)
    n_incomplete = int((~has_space_elements).sum())
    subset = subset[has_space_elements].copy()
    if n_incomplete:
        print(f'{system_key}: excluded {n_incomplete:,} samples missing an element named in material_space.')
    coords, nearest, nearest_intent, distances, oxy = [], [], [], [], []
    for comp in subset['composition']:
        vec = np.array([comp.get(el, 0.0) for el in system], dtype=float)
        total = vec.sum()
        if total <= 0:
            coords.append(None); nearest.append(None); nearest_intent.append(None); distances.append(np.nan); oxy.append(comp.get('O', np.nan)); continue
        vec /= total
        dists = {r['label']: float(np.linalg.norm(vec - np.array([r['phase_comp'][el] for el in system]))) for r in refs}
        label = min(dists, key=dists.get)
        coords.append(vec); nearest.append(label); nearest_intent.append(ref_by_label[label]['intent']); distances.append(dists[label]); oxy.append(comp.get('O', np.nan))
    subset['coords'] = coords
    subset['nearest_phase'] = nearest
    subset['nearest_intent'] = nearest_intent
    subset['nearest_distance'] = distances
    subset['oxygen_at_fraction'] = subset['normalized_composition'].map(
        lambda c: c.get('O', np.nan) if 'O' in c else np.nan
    )
    subset = subset[subset['coords'].notna()].copy()
    if subset.empty:
        print(f'No measured sample compositions map to {system_key}.')
        continue

    # Build a triangular barycentric grid and assign each node to its nearest formula.
    basin = {r['label']: [[], [], []] for r in refs}
    for i in range(GRID_DIVISIONS + 1):
        for j in range(GRID_DIVISIONS + 1 - i):
            vec = np.array([i, j, GRID_DIVISIONS - i - j], dtype=float) / GRID_DIVISIONS
            label = min(refs, key=lambda r: np.linalg.norm(vec - np.array([r['phase_comp'][el] for el in system])))['label']
            for axis, value in zip(basin[label], vec):
                axis.append(100 * value)

    fig = go.Figure()
    for label, (a, b, c) in basin.items():
        if a:
            fig.add_trace(go.Scatterternary(
                a=a, b=b, c=c, mode='markers', name=f'Nearest-composition region: {label}',
                marker={'size': 4, 'color': colors[label], 'opacity': 0.16},
                hoverinfo='skip', showlegend=False,
            ))
    for label, ref in ref_by_label.items():
        group = subset[subset['nearest_phase'] == label]
        for intent in group['nearest_intent'].dropna().unique():
            points = group[group['nearest_intent'] == intent]
            if points.empty:
                continue
            symbol = 'circle' if intent.startswith('Aimed') else ('diamond' if intent.startswith('Unplanned') else 'square')
            fig.add_trace(go.Scatterternary(
                a=[100 * x[0] for x in points['coords']],
                b=[100 * x[1] for x in points['coords']],
                c=[100 * x[2] for x in points['coords']],
                mode='markers', name=f'Samples nearest {label} ({intent})',
                text=points['lab_id'].fillna(points['entry_id']),
                customdata=np.column_stack([points['oxygen_at_fraction'], points['nearest_distance']]),
                marker={'size': 5, 'color': colors[label], 'symbol': symbol,
                        'opacity': 0.75, 'line': {'color': 'black', 'width': 0.3}},
                hovertemplate='%{text}<br>nearest: ' + label + '<br>O at.frac: %{customdata[0]:.3f}'
                              + '<br>distance to formula: %{customdata[1]:.3f}<br>'
                              + f'{system[0]}: %{{a:.1f}}%<br>{system[1]}: %{{b:.1f}}%<br>{system[2]}: %{{c:.1f}}%<extra></extra>',
            ))
    fig.add_trace(go.Scatterternary(
        a=[100 * r['phase_comp'][system[0]] for r in refs],
        b=[100 * r['phase_comp'][system[1]] for r in refs],
        c=[100 * r['phase_comp'][system[2]] for r in refs],
        mode='markers+text', name='Candidate phase formulas', text=[r['label'] for r in refs],
        textposition='top center',
        marker={'size': 13, 'symbol': 'x', 'color': 'black', 'line': {'width': 2}},
        hovertemplate='%{text}<br>candidate formula (stability unverified)<br>'
                      + f'{system[0]}: %{{a:.1f}}%<br>{system[1]}: %{{b:.1f}}%<br>{system[2]}: %{{c:.1f}}%<extra></extra>',
    ))
    axis = lambda el: {'title': {'text': f'{el} (at.%)'}, 'min': 0, 'dtick': 25,
                       'showline': True, 'linecolor': 'black', 'linewidth': 1.5,
                       'ticks': 'outside', 'showgrid': True, 'gridcolor': 'grey'}
    fig.update_layout(
        ternary={'sum': 100, 'aaxis': axis(system[0]), 'baxis': axis(system[1]), 'caxis': axis(system[2])},
        title=f'{system_key}: samples and nearest-reference composition regions',
        width=850, height=760, legend={'orientation': 'h', 'y': -0.15},
        template='plotly_white',
    )
    figures[system_key] = fig
    fig.show(config=PLOTLY_CONFIG)

    nearest_summary = subset.groupby(['nearest_phase', 'nearest_intent'], dropna=False).agg(
        n_samples=('entry_id', 'size'), n_lab_ids=('lab_id', 'nunique'),
        median_distance=('nearest_distance', 'median'), median_oxygen=('oxygen_at_fraction', 'median'),
    ).reset_index()
    display(nearest_summary)
    family_phase = subset.dropna(subset=['lab_id_family']).groupby(
        ['lab_id_family', 'nearest_phase', 'nearest_intent'], as_index=False
    ).size().rename(columns={'size': 'n_samples_in_family'})
    family_phase['family_total_samples'] = family_phase.groupby('lab_id_family')['n_samples_in_family'].transform('sum')
    family_phase['family_fraction_nearest_phase'] = family_phase['n_samples_in_family'] / family_phase['family_total_samples']
    dominant_family_phase = family_phase.loc[family_phase.groupby('lab_id_family')['family_fraction_nearest_phase'].idxmax()]
    print(f'{system_key}: inferred-family majority assignment (requires confirmation of lab-ID grouping):')
    display(dominant_family_phase.groupby(['nearest_phase', 'nearest_intent']).agg(
        n_inferred_families=('lab_id_family', 'nunique'),
        median_family_fraction=('family_fraction_nearest_phase', 'median'),
    ).reset_index())

## Interpretation / inputs needed for a second round

- The supplied CIF formulas are treated as user-designated stable references; identify any CIF that should not be included in that reference set. The Zr2SeN2 reference formula was supplied separately, without a CIF.
- Confirm that the aimed/unplanned labels are correct and whether more reference phases or material spaces should be included.
- Provide the specific statement/prediction from Vladan's theory and which measurable outcome it predicts. Until then, `VLADAN_THEORY_MAPPING` remains unset.
- Compare library/process-level aggregates and replicate libraries before inferential statistics. The per-sample plots are descriptive because neighboring positions within a combinatorial library are spatially correlated.
- Oxygen/anion fractions come from `elemental_composition` normalized over reported components. Samples missing any element in the reference formula are excluded; oxygen must also be reported. The fraction-sum sensitivity excludes totals outside 0.9-1.1. Because atomic fractions are compositional, interpret their correlation alongside the matrix-normalized ratio sensitivity, not as independent proof of oxygen substitution.
- Similarity distance and nearest-reference regions ignore crystal structure, phase stability, synthesis kinetics, uncertainty, and cation/anion site occupancies. They are visual aids, not proof of attraction basins or thermodynamic preference.

Plotly charts are interactive; use the camera toolbar button to export SVG.

## Literature context

- S. Aliramaji et al., “Impurity sources and incorporation pathways during sputter deposition of Mg and Al thin films,” *Surface and Coatings Technology* (2025), [doi:10.1016/j.surfcoat.2025.132216](https://doi.org/10.1016/j.surfcoat.2025.132216). This systematic study compares base pressure, Ar purity and deposition rate; it illustrates that residual-gas flux and rate can matter, while effects vary by material and a simple proxy does not identify the species.
- S. Aliramaji et al., “Effect of Growth Temperature and Atmosphere Exposure Time on Impurity Incorporation in Sputtered Mg, Al, and Ca Thin Films,” *Materials* 16, 414 (2023), [doi:10.3390/ma16010414](https://doi.org/10.3390/ma16010414). It separates growth from post-deposition oxidation and reports morphology/temperature and air-exposure effects—important because EDX timestamps do not establish exposure history.
- “Effect of background gas environment on oxygen incorporation in TiN films deposited using UHV reactive magnetron sputtering,” *Vacuum* (2009), [ScienceDirect record](https://www.sciencedirect.com/science/article/pii/S0042207X08002340). The study reports oxygen incorporation under poorer background-vacuum conditions; TiN behavior should not be assumed quantitative for these chalcogenide systems.
- “Oxygen incorporation in Al thin films during deposition by dc magnetron sputtering,” *Vacuum* 38 (1988), [ScienceDirect record](https://www.sciencedirect.com/science/article/pii/0042207X88905660). It reports dependencies on discharge power and substrate temperature, showing why those are plausible covariates rather than universal effect directions.
- L. C. Gomes, A. Carvalho, and A. H. Castro Neto, “Vacancies and oxidation of 2D group-IV monochalcogenides,” *Physical Review B* 94, 054103 (2016), [doi:10.1103/PhysRevB.94.054103](https://doi.org/10.1103/PhysRevB.94.054103). Their calculations support oxygen substitution on chalcogen sites in monolayer SnS/SnSe/GeS/GeSe; this is mechanistic analogy, not direct evidence for bulk SnPS3 or Zr-Se-N.
- “Identifying substitutional oxygen as a prolific point defect in monolayer transition metal dichalcogenides with experiment and theory,” *Nature Communications* 10, 3382 (2019), [doi:10.1038/s41467-019-11342-2](https://doi.org/10.1038/s41467-019-11342-2). The evidence is from 2D TMDs, not the phosphosulfide/nitride systems analyzed here.
- R. Woods-Robinson et al., “Role of disorder in the synthesis of metastable zinc zirconium nitrides,” *Physical Review Materials* 6, 043804 (2022), [doi:10.1103/PhysRevMaterials.6.043804](https://doi.org/10.1103/PhysRevMaterials.6.043804). This sputtering study reports disorder-tolerant metastable polymorphs in Zn-Zr-N and includes Vladan Stevanović; it may be the relevant theory context, but its specific prediction must be confirmed before treating Exp9 as a test of it.
- Materials Project, [Phase Diagrams methodology](https://docs.materialsproject.org/methodology/materials-methodology/thermodynamic-stability/phase-diagrams-pds): thermodynamic stability is evaluated with energies/free energies and a convex hull, not composition-space proximity alone.

# Oxygen versus local deposition rate from EDX

Each plotted point pairs the oxygen atomic fraction and fitted layer thickness from the **same EDX result**. Its rate is that thickness divided by the recorded deposition time of the uniquely linked sputtering run. Base pressure is shared by the run, not measured at each point. The second plot uses base pressure / local rate as a qualitative residual-gas exposure proxy; base pressure is not oxygen partial pressure during deposition.

EDX thickness depends on an assumed density and may be coupled to fitted composition. Oxygen may also include surface oxidation or contamination after deposition. These plots cannot distinguish those mechanisms. Positions within one EDX map and libraries from the same run are correlated; map- and run-level summaries below avoid interpreting every point as an independent experiment. A direct sputtering-to-library reference does not prove that the sample was never subsequently processed.

This notebook reads only records visible to the authenticated account and works locally or in NORTH with the accompanying `oasis_auth.py`. Archive values use SI units; displayed rates are nm/min and base pressures Torr.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'notebooks' / 'oasis_trends'):
    if (candidate / 'oasis_auth.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Start Jupyter in the repository or notebooks/oasis_trends folder.')

from collections import Counter, defaultdict
import re
import numpy as np
import pandas as pd
import plotly.express as px
import requests
from oasis_auth import api_base_url, get_token

API_BASE_URL = api_base_url()
PLOTLY_CONFIG = {'toImageButtonOptions': {'format': 'svg', 'scale': 1}}
PA_PER_TORR = 101325 / 760
FRACTION_SUM_RANGE = (0.9, 1.1)
session = requests.Session()
token = get_token()
if not token:
    raise RuntimeError('No NOMAD token found; see notebooks/oasis_trends/README.md.')
session.headers.update({'Authorization': f'Bearer {token}', 'Accept': 'application/json'})
del token
print('API base URL:', API_BASE_URL)

def pages(endpoint, query, required, page_size):
    body = {'owner': 'visible', 'query': query, 'required': required,
            'pagination': {'page_size': page_size, 'order_by': 'entry_id', 'order': 'asc'}}
    rows, seen, total = [], set(), None
    while True:
        response = session.post(API_BASE_URL + endpoint, json=body, timeout=180, allow_redirects=False)
        response.raise_for_status()
        payload = response.json()
        total = payload.get('pagination', {}).get('total', total)
        page = payload.get('data', [])
        rows.extend(page)
        cursor = payload.get('pagination', {}).get('next_page_after_value')
        if not cursor:
            break
        if not page or cursor in seen:
            raise RuntimeError(f'Incomplete pagination at {endpoint}.')
        seen.add(cursor)
        body['pagination']['page_after_value'] = cursor
    if total is not None and len(rows) != total:
        raise RuntimeError(f'{endpoint}: fetched {len(rows)} of {total} visible records.')
    print(f'{endpoint}: fetched {len(rows)}; visible total {total}')
    return rows

def schema_query(name):
    return {'section_defs.definition_qualified_name:all': [
        'nomad_dtu_nanolab_plugin.schema_packages.' + name]}

def selected_archives(schema, fields, page_size=50):
    entries = pages('/entries/archive/query', schema_query(schema),
                    {'include': ['entry_id', 'upload_id'], 'data': fields}, page_size)
    archives = []
    for entry in entries:
        archive = entry.get('archive') or {}
        data = archive.get('data')
        if not isinstance(data, dict):
            raise RuntimeError(f"Missing archive data for {entry['entry_id']}.")
        upload_id = (archive.get('metadata') or {}).get('upload_id') or entry.get('upload_id')
        archives.append((entry['entry_id'], upload_id, data))
    return archives

## Fetch and resolve references

Only single-library EDX entries linked by entry reference to exactly one sputtering run are used. Relative mainfile references are resolved with visible library metadata; lab-ID similarity is not used as a fallback. Runs must have a positive deposition time and exactly one step marked as creating a new thin film. RTP input references are flagged as a conservative sensitivity check, not as proof that a given EDX measurement was annealed.

In [ ]:
libraries = pages('/entries/query', schema_query('sample.DTUCombinatorialLibrary'),
                  {'include': ['entry_id', 'upload_id', 'mainfile']}, 200)
mainfiles = {(item['upload_id'], item['mainfile']): item['entry_id'] for item in libraries}

def ref_id(ref, own_upload):
    if not isinstance(ref, str):
        return None
    match = re.search(r'/uploads/([^/]+)/archive/mainfile/([^#]+)', ref)
    if match:
        return mainfiles.get((match.group(1), match.group(2)))
    match = re.search(r'/archive/mainfile/([^#]+)', ref)
    if match:
        return mainfiles.get((own_upload, match.group(1)))
    match = re.search(r'/archive/([A-Za-z0-9_-]{20,})(?:[#/]|$)', ref)
    if not match:
        match = re.search(r'/entries/([A-Za-z0-9_-]{20,})/archive', ref)
    return match.group(1) if match else None

process_parameter_names = [
    'deposition_time', 'material_space', 'deposition_temperature',
    'deposition_temperature_2', 'deposition_temperature_setpoint',
    'sputter_pressure', 'ar_flow', 'h2s_in_ar_flow', 'nh3_in_ar_flow',
    'ph3_in_ar_flow', 'n2_flow', 'o2_in_ar_flow',
    'ar_partial_pressure', 'h2s_partial_pressure', 'nh3_partial_pressure',
    'ph3_partial_pressure', 'n2_partial_pressure', 'o2_partial_pressure',
]
source_parameter_names = ['target_material', 'power_type', 'target_usage', 'applied_power', 'average_voltage', 'std_voltage']
source_parameter_names += ['target_id']
parameter_fields = {name: '*' for name in process_parameter_names}
parameter_fields.update({gun: {name: '*' for name in source_parameter_names}
                         for gun in ('magkeeper3', 'magkeeper4', 'taurus')})
parameter_fields['s_cracker'] = {'zone1_temperature': '*', 'zone2_temperature': '*',
                                 'zone3_temperature': '*', 'valve_on_time': '*', 'valve_frequency': '*'}
run_fields = {'lab_id': '*', 'datetime': '*', 'base_pressure': '*', 'sulfur_partial_pressure': '*',
              'deposition_parameters': parameter_fields,
              'source_presput': {'target_name': '*', 'presput_time': '*', 'presput_power': '*',
                                 'presput_pressure': '*', 'presput_ar_flow': '*'},
              'source_ramp_up': {'target_name': '*', 'plasma_ignition_power': '*', 'plasma_ignition_pressure': '*'},
              'source_deprate': {'target_name': '*', 'source_deprate': '*', 'source_deprate_ref_mat': '*'},
              'samples': '*', 'steps': {'name': '*', 'creates_new_thin_film': '*'}}
edx_fields = {'lab_id': '*', 'datetime': '*', 'samples': '*',
              'results': {'name': '*', 'x_absolute': '*', 'y_absolute': '*',
                          'layer_thickness': '*', 'assumed_material_density': '*',
                          'quantifications': '*'}}
rtp_fields = {'input_samples': {'input_combi_lib': '*'}}
runs_data = selected_archives('sputtering.DTUSputtering', run_fields)
edx_data = selected_archives('edx.EDXMeasurement', edx_fields, page_size=25)
rtp_data = selected_archives('rtp.DtuRTP', rtp_fields)

## Construct paired measurements

Keep individual results with positive fitted thickness, a reported oxygen atomic fraction in [0, 1], and a sum of reported elemental fractions within the configured quality interval. Missing oxygen is not interpreted as zero. Multiple EDX maps of a library remain visible, with their separate entry IDs and timestamps; they are not assumed independent.

In [ ]:
def finite(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return np.nan
    return number if np.isfinite(number) else np.nan

diagnostics = Counter()
run_info, library_runs = {}, defaultdict(set)
for entry_id, upload_id, data in runs_data:
    parameters = data.get('deposition_parameters') or {}
    steps = data.get('steps') or []
    run_info[entry_id] = {
        'run_id': entry_id, 'run_lab_id': data.get('lab_id'),
        'run_time': data.get('datetime'), 'seconds': finite(parameters.get('deposition_time')),
        'space': parameters.get('material_space') or 'Unknown',
        'pressure_torr': finite(data.get('base_pressure')) / PA_PER_TORR,
        'n_deposition_steps': sum(s.get('creates_new_thin_film') is True for s in steps),
    }
    for sample in data.get('samples') or []:
        library_id = ref_id(sample.get('reference'), upload_id)
        diagnostics['run sample references'] += 1
        if library_id:
            library_runs[library_id].add(entry_id)
        else:
            diagnostics['unresolved run sample references'] += 1

rtp_inputs = set()
for _, upload_id, data in rtp_data:
    for mounting in data.get('input_samples') or []:
        library_id = ref_id(mounting.get('input_combi_lib'), upload_id)
        if library_id:
            rtp_inputs.add(library_id)
        else:
            diagnostics['unresolved RTP input references'] += 1

point_rows = []
for edx_id, upload_id, data in edx_data:
    sample_refs = data.get('samples') or []
    if len(sample_refs) != 1:
        diagnostics['EDX entries without exactly one sample'] += 1
        continue
    library_id = ref_id(sample_refs[0].get('reference'), upload_id)
    if not library_id:
        diagnostics['EDX entries with unresolved sample reference'] += 1
        continue
    matching_runs = library_runs.get(library_id, set())
    if len(matching_runs) != 1:
        diagnostics['EDX entries without exactly one linked run'] += 1
        continue
    run = run_info[next(iter(matching_runs))]
    if not run['seconds'] > 0:
        diagnostics['EDX entries without positive deposition time'] += 1
        continue
    if run['n_deposition_steps'] != 1:
        diagnostics['EDX entries without exactly one deposition step'] += 1
        continue
    for index, result in enumerate(data.get('results') or []):
        diagnostics['EDX results considered'] += 1
        quantifications = result.get('quantifications') or []
        composition = {q.get('element'): finite(q.get('atomic_fraction'))
                       for q in quantifications if q.get('element')}
        thickness = finite(result.get('layer_thickness'))
        oxygen = composition.get('O', np.nan)
        total = sum(v for v in composition.values() if np.isfinite(v))
        if not (thickness > 0 and 0 <= oxygen <= 1 and FRACTION_SUM_RANGE[0] <= total <= FRACTION_SUM_RANGE[1]):
            diagnostics['EDX results excluded by thickness, oxygen or fraction-sum filter'] += 1
            continue
        thickness_nm = thickness * 1e9
        rate = thickness_nm * 60 / run['seconds']
        pressure = run['pressure_torr']
        point_rows.append({
            **run, 'library_id': library_id, 'library_lab_id': sample_refs[0].get('lab_id'),
            'edx_id': edx_id, 'edx_lab_id': data.get('lab_id'), 'edx_time': data.get('datetime'),
            'result_index': index, 'point_name': result.get('name'),
            'x_mm': finite(result.get('x_absolute')) * 1000,
            'y_mm': finite(result.get('y_absolute')) * 1000,
            'thickness_nm': thickness_nm, 'oxygen': oxygen, 'fraction_sum': total,
            'density_kg_m3': finite(result.get('assumed_material_density')),
            'rate_nm_min': rate,
            'pressure_per_rate': pressure / rate if pressure > 0 else np.nan,
            'rtp_input_any_time': library_id in rtp_inputs,
        })

points = pd.DataFrame(point_rows)
display(pd.DataFrame(list(diagnostics.items()), columns=['diagnostic', 'count']))
if points.empty:
    raise RuntimeError('No paired EDX points passed the documented filters; inspect the diagnostics above.')
print(f"Points: {len(points):,}; EDX maps: {points['edx_id'].nunique():,}; "
      f"libraries: {points['library_id'].nunique():,}; runs: {points['run_id'].nunique():,}.")
print(f"Points with positive base pressure: {points['pressure_per_rate'].notna().sum():,}; "
      f"points from libraries ever listed as RTP input: {points['rtp_input_any_time'].sum():,}.")
display(points[['rate_nm_min', 'oxygen', 'pressure_torr', 'fraction_sum']].describe())

## Interactive point-by-point plots

Hover shows the run, library, EDX entry, result index, position, thickness, shared deposition time, and base pressure. Use the legend to isolate material spaces, or set the plot-only filters below to focus on one run or material space. Use the Plotly camera button to export SVG.

In [ ]:
PLOT_RUN_ID = None  # optional exact run entry ID; does not change statistical summaries
PLOT_MATERIAL_SPACE = None  # optional exact material-space string
plot_points = points.copy()
if PLOT_RUN_ID is not None:
    plot_points = plot_points[plot_points['run_id'] == PLOT_RUN_ID]
if PLOT_MATERIAL_SPACE is not None:
    plot_points = plot_points[plot_points['space'] == PLOT_MATERIAL_SPACE]
if plot_points.empty:
    raise ValueError('No points match the plot filters; check the run ID and material space.')
print(f'Plotting {len(plot_points):,} paired EDX points.')
hover = ['run_lab_id', 'run_id', 'library_lab_id', 'library_id', 'edx_lab_id', 'edx_id',
         'run_time', 'edx_time', 'result_index', 'point_name', 'x_mm', 'y_mm', 'thickness_nm',
         'seconds', 'pressure_torr', 'density_kg_m3', 'fraction_sum', 'rtp_input_any_time']
labels = {'rate_nm_min': 'Local EDX-derived deposition rate (nm/min)',
          'oxygen': 'EDX oxygen atomic fraction', 'space': 'Sputtering material space',
          'pressure_per_rate': 'Base pressure / local rate (Torr min/nm)'}
rate_figure = px.scatter(plot_points, x='rate_nm_min', y='oxygen', color='space',
                         hover_data=hover, labels=labels, log_x=True, opacity=0.55,
                         template='plotly_white', title='EDX oxygen versus local deposition rate')
rate_figure.update_xaxes(showline=True, mirror=True, linecolor='black')
rate_figure.update_yaxes(showline=True, mirror=True, linecolor='black')
rate_figure.show(config=PLOTLY_CONFIG)

pressure_points = plot_points[plot_points['pressure_per_rate'].notna()]
if pressure_points.empty:
    print('No points have a positive base pressure; pressure/rate plot is unavailable.')
else:
    pressure_figure = px.scatter(pressure_points, x='pressure_per_rate', y='oxygen', color='space',
                                 hover_data=hover + ['rate_nm_min'], labels=labels, log_x=True,
                                 opacity=0.55, template='plotly_white',
                                 title='EDX oxygen versus base pressure / local deposition rate')
    pressure_figure.update_xaxes(showline=True, mirror=True, linecolor='black')
    pressure_figure.update_yaxes(showline=True, mirror=True, linecolor='black')
    pressure_figure.show(config=PLOTLY_CONFIG)

## Within-map and run-level sensitivity

Within a single map the time and base pressure are constant, so a rate–oxygen correlation is also a thickness–oxygen correlation; it cannot establish a deposition-rate mechanism. Calculate Spearman correlations only for maps with at least three valid points and variation in both quantities. Summarize map correlations by run before reporting an overall median. The RTP-input exclusion is deliberately conservative and does not determine the actual sequence of events.

In [ ]:
def map_spearman(group):
    if len(group) < 3 or group['rate_nm_min'].nunique() < 2 or group['oxygen'].nunique() < 2:
        return np.nan
    return group['rate_nm_min'].corr(group['oxygen'], method='spearman')

map_rows = []
for (run_id, edx_id), group in points.groupby(['run_id', 'edx_id']):
    map_rows.append({'run_id': run_id, 'edx_id': edx_id, 'space': group['space'].iloc[0],
                     'n_points': len(group), 'map_rho': map_spearman(group),
                     'rtp_input_any_time': group['rtp_input_any_time'].iloc[0]})
map_stats = pd.DataFrame(map_rows)
run_stats = map_stats.groupby(['run_id', 'space'], as_index=False).agg(
    n_maps=('edx_id', 'nunique'), median_map_rho=('map_rho', 'median'))
print(f"Maps with defined correlation: {map_stats['map_rho'].notna().sum()} / {len(map_stats)}")
if run_stats['median_map_rho'].notna().any():
    print(f"Median of run-level median map correlations: {run_stats['median_map_rho'].median():.3f}")
else:
    print('No maps have enough variation for a within-map correlation.')
clean_maps = map_stats[~map_stats['rtp_input_any_time']]
clean_runs = clean_maps.groupby('run_id')['map_rho'].median()
print(f"Excluding maps whose library is ever an RTP input: {len(clean_runs)} runs; "
      f"{clean_runs.notna().sum()} runs have a defined correlation.")
if clean_runs.notna().any():
    print(f"Median run correlation after RTP-input exclusion: {clean_runs.median():.3f}")
display(run_stats.groupby('space').agg(
    n_runs=('run_id', 'nunique'), n_runs_with_correlation=('median_map_rho', 'count'),
    median_run_correlation=('median_map_rho', 'median')).sort_values('n_runs', ascending=False))
display(map_stats.head(20))

# First weight repeated EDX maps equally within each library, then libraries equally within each run.
map_means = points.groupby(['run_id', 'library_id', 'edx_id'], as_index=False).agg(
    mean_oxygen=('oxygen', 'mean'), mean_rate_nm_min=('rate_nm_min', 'mean'))
library_means = map_means.groupby(['run_id', 'library_id'], as_index=False).agg(
    mean_oxygen=('mean_oxygen', 'mean'), mean_rate_nm_min=('mean_rate_nm_min', 'mean'))
run_means = library_means.groupby('run_id', as_index=False).agg(
    mean_oxygen=('mean_oxygen', 'mean'), mean_rate_nm_min=('mean_rate_nm_min', 'mean'),
    n_libraries=('library_id', 'nunique'))
context = points[['run_id', 'run_lab_id', 'space', 'pressure_torr']].drop_duplicates('run_id')
run_means = run_means.merge(context, on='run_id', validate='one_to_one')
run_means['pressure_per_rate'] = run_means['pressure_torr'].where(run_means['pressure_torr'] > 0) / run_means['mean_rate_nm_min']
def run_correlations(group):
    return pd.Series({
        'runs': len(group),
        'runs_with_pressure': group['pressure_per_rate'].notna().sum(),
        'rate_oxygen_rho': group['mean_rate_nm_min'].corr(group['mean_oxygen'], method='spearman'),
        'pressure_oxygen_rho': group['pressure_torr'].where(group['pressure_torr'] > 0).corr(group['mean_oxygen'], method='spearman'),
        'pressure_per_rate_oxygen_rho': group['pressure_per_rate'].corr(group['mean_oxygen'], method='spearman'),
    })
print('Run-level correlations (unadjusted and exploratory):')
display(run_correlations(run_means).to_frame('all runs'))
display(pd.DataFrame([{'space': space, **run_correlations(group).to_dict()}
                      for space, group in run_means.groupby('space') if len(group) >= 5]))

run_hover = ['run_lab_id', 'run_id', 'n_libraries', 'pressure_torr']
run_labels = {'mean_rate_nm_min': 'Mean EDX-derived rate (nm/min)',
              'mean_oxygen': 'Mean EDX oxygen atomic fraction', 'space': 'Sputtering material space',
              'pressure_per_rate': 'Base pressure / mean rate (Torr min/nm)'}
run_rate_figure = px.scatter(run_means, x='mean_rate_nm_min', y='mean_oxygen', color='space',
                             hover_data=run_hover, labels=run_labels, log_x=True,
                             template='plotly_white', title='Run-level oxygen versus mean deposition rate')
run_rate_figure.update_xaxes(showline=True, mirror=True, linecolor='black')
run_rate_figure.update_yaxes(showline=True, mirror=True, linecolor='black')
run_rate_figure.show(config=PLOTLY_CONFIG)
run_pressure = run_means[run_means['pressure_per_rate'].notna()]
if run_pressure.empty:
    print('No runs have positive base pressure; run-level pressure/rate plot is unavailable.')
else:
    run_pressure_figure = px.scatter(run_pressure, x='pressure_per_rate', y='mean_oxygen', color='space',
                                    hover_data=run_hover + ['mean_rate_nm_min'], labels=run_labels, log_x=True,
                                    template='plotly_white', title='Run-level oxygen versus pressure / rate')
    run_pressure_figure.update_xaxes(showline=True, mirror=True, linecolor='black')
    run_pressure_figure.update_yaxes(showline=True, mirror=True, linecolor='black')
    run_pressure_figure.show(config=PLOTLY_CONFIG)

## Oxygen versus sputtering-process parameters

Correlations use one oxygen mean per eligible run (points averaged within maps, maps within libraries, libraries within runs), not individual spatial points. Sputtering parameters are joined by the exact run ID already used above. Coverage is pairwise, with missing values left missing; no blanket imputation. A pair needs at least 10 runs and variation in both fields. The material-space table uses at least 8 runs per pair; very small groups are descriptive only. The pooled matrix can be dominated by material differences; the within-material rank-residual sensitivity removes each space's mean rank for each pair and requires at least 3 usable runs per contributing material. These are exploratory associations, not causal rankings. Many tests, correlated parameters, changing missingness, authors, dates, sample treatment, and process history can confound results.

Sputtering literature motivates checks of residual-gas/base pressure, working-gas flow and purity, deposition rate, target conditioning and power, substrate temperature/morphology, and post-deposition atmosphere exposure. These determine different impurity pathways and are often correlated by recipe. The EDX cohort is selected for positive fitted thickness and deposition time and may not represent every sputtered sample; RTP-input exclusion below is conservative, not proof of measurement timing. EDX oxygen and thickness are jointly fitted. Source quantities refer to physical gun positions, not a fixed element; target material can change. Presput values are matched by exact gun name; repeated presput stages are summed in duration and duration-weighted for power, pressure and flow. Missing values remain missing, not zero. Cracker section presence is a recording proxy, not proof of use or non-use. Target and power-type group comparisons are descriptive and include counts and material-space strata. Temperature and pressure records may contain schema defaults. Gas partial pressures are calculated from recorded flows and total pressure, not direct measurements of residual oxygen; recorded O2 flow is the 20%-O2-in-Ar line setting.

In [ ]:
MIN_CORRELATION_RUNS = 10
MIN_SYSTEM_RUNS = 8
GUNS = ('magkeeper3', 'magkeeper4', 'taurus')

def process_record(entry_id, data):
    params = data.get('deposition_parameters') or {}
    row = {'run_id': entry_id}
    row['deposition_time_min'] = finite(params.get('deposition_time')) / 60
    row['base_pressure_torr'] = finite(data.get('base_pressure')) / PA_PER_TORR
    row['sulfur_partial_pressure_torr'] = finite(data.get('sulfur_partial_pressure')) / PA_PER_TORR
    for key in ('deposition_temperature', 'deposition_temperature_2', 'deposition_temperature_setpoint'):
        row[key + '_C'] = finite(params.get(key)) - 273.15
    for key in ('sputter_pressure', 'ar_partial_pressure', 'h2s_partial_pressure',
                'nh3_partial_pressure', 'ph3_partial_pressure', 'n2_partial_pressure',
                'o2_partial_pressure'):
        row[key + '_torr'] = finite(params.get(key)) / PA_PER_TORR
    for key in ('ar_flow', 'h2s_in_ar_flow', 'nh3_in_ar_flow', 'ph3_in_ar_flow',
                'n2_flow', 'o2_in_ar_flow'):
        row[key + '_sccm'] = finite(params.get(key)) * 60e6
    for gun in GUNS:
        data_source = params.get(gun) or {}
        row[gun + '_target'] = data_source.get('target_material')
        target_ref = data_source.get('target_id') or {}
        row[gun + '_target_id'] = target_ref.get('lab_id') if isinstance(target_ref, dict) else None
        row[gun + '_power_type'] = data_source.get('power_type')
        row[gun + '_power_W'] = finite(data_source.get('applied_power'))
        row[gun + '_voltage_V'] = finite(data_source.get('average_voltage'))
        row[gun + '_voltage_std_V'] = finite(data_source.get('std_voltage'))
        row[gun + '_target_usage_kWh'] = finite(data_source.get('target_usage'))
        matches = [item for item in (data.get('source_presput') or []) if item.get('target_name') == gun]
        durations = np.array([finite(item.get('presput_time')) for item in matches])
        row[gun + '_presput_records'] = len(matches)
        row[gun + '_presput_recorded'] = 'Recorded' if matches else 'Not recorded'
        row[gun + '_presput_time_min'] = np.nansum(durations) / 60 if np.isfinite(durations).any() else np.nan
        def duration_weighted_presput(key, scale):
            values = np.array([finite(item.get(key)) for item in matches]) * scale
            valid = np.isfinite(values) & np.isfinite(durations) & (durations > 0)
            return np.average(values[valid], weights=durations[valid]) if valid.any() else np.nan
        row[gun + '_presput_power_W'] = duration_weighted_presput('presput_power', 1)
        row[gun + '_presput_pressure_torr'] = duration_weighted_presput('presput_pressure', 1 / PA_PER_TORR)
        row[gun + '_presput_ar_flow_sccm'] = duration_weighted_presput('presput_ar_flow', 60e6)
        ramp = [item for item in (data.get('source_ramp_up') or []) if item.get('target_name') == gun]
        def finite_mean(items, key, scale):
            values = np.array([finite(item.get(key)) * scale for item in items])
            values = values[np.isfinite(values)]
            return values.mean() if len(values) else np.nan
        row[gun + '_ignition_power_W'] = finite_mean(ramp, 'plasma_ignition_power', 1)
        row[gun + '_ignition_pressure_torr'] = finite_mean(ramp, 'plasma_ignition_pressure', 1 / PA_PER_TORR)
        deprate = [item for item in (data.get('source_deprate') or []) if item.get('target_name') == gun]
        row[gun + '_source_deprate_nm_min'] = finite_mean(deprate, 'source_deprate', 60e9)
        row[gun + '_source_deprate_records'] = len(deprate)
    cracker = params.get('s_cracker')
    row['cracker_section_recorded'] = 'Recorded' if isinstance(cracker, dict) else 'Not recorded'
    cracker = cracker or {}
    for zone in (1, 2, 3):
        row[f'cracker_zone{zone}_C'] = finite(cracker.get(f'zone{zone}_temperature')) - 273.15
    row['cracker_valve_on_s'] = finite(cracker.get('valve_on_time'))
    row['cracker_valve_frequency_Hz'] = finite(cracker.get('valve_frequency'))
    for name in ('base_pressure_torr', 'sputter_pressure_torr'):
        if row[name] <= 0:
            row[name] = np.nan
    for name in row:
        if (name.endswith('_flow_sccm') or name.endswith('_partial_pressure_torr')) and row[name] < 0:
            row[name] = np.nan
    return row

process_data = pd.DataFrame([process_record(entry_id, data) for entry_id, _, data in runs_data])
if process_data['run_id'].duplicated().any():
    raise ValueError('Duplicate sputtering run IDs in process parameters.')
process_runs = run_means.merge(process_data, on='run_id', how='left', validate='one_to_one', indicator=True)
print('Eligible oxygen runs:', len(process_runs), '; linked process records:',
      int((process_runs['_merge'] == 'both').sum()))
process_runs = process_runs.drop(columns='_merge')
process_runs['EDX_rate_nm_min'] = process_runs['mean_rate_nm_min']
process_runs['EDX_thickness_nm'] = process_runs['mean_rate_nm_min'] * process_runs['deposition_time_min']
categorical_columns = ['cracker_section_recorded'] + [g + suffix for g in GUNS
    for suffix in ('_target', '_target_id', '_power_type', '_presput_recorded')]
parameter_columns = [c for c in process_data.columns
                     if c != 'run_id' and c not in categorical_columns and not c.endswith('_records')]
parameter_columns += ['EDX_rate_nm_min', 'EDX_thickness_nm']
numeric = ['mean_oxygen'] + parameter_columns
availability = pd.DataFrame({'parameter': parameter_columns,
    'n_reported': [process_runs[c].notna().sum() for c in parameter_columns],
    'n_unique': [process_runs[c].nunique() for c in parameter_columns]})
print('Parameter coverage among eligible oxygen runs:')
display(availability)
print('Target materials recorded at each source (including missing values):')
for gun in GUNS:
    display(process_runs.groupby(gun + '_target', dropna=False).size().rename('runs').to_frame())
categorical_summaries = []
for col in categorical_columns:
    for (space, category), group in process_runs.dropna(subset=[col]).groupby(['space', col]):
        categorical_summaries.append({'space': space, 'parameter': col, 'category': category,
                                      'runs': len(group), 'median_oxygen': group['mean_oxygen'].median()})
categorical_process_summary = pd.DataFrame(categorical_summaries)
print('Categorical summaries within material space; presence or absence is not a causal comparison:')
display(categorical_process_summary)

def rho_pair(group, x, y, minimum):
    pair = group[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(pair) < minimum or pair[x].nunique() < 2 or pair[y].nunique() < 2:
        return len(pair), np.nan
    return len(pair), pair[x].corr(pair[y], method='spearman')

def within_space_rho(frame, x):
    usable = frame[['space', x, 'mean_oxygen']].replace([np.inf, -np.inf], np.nan).dropna()
    ranks = []
    for _, group in usable.groupby('space'):
        if len(group) < 3 or group[x].nunique() < 2 or group['mean_oxygen'].nunique() < 2:
            continue
        xrank = group[x].rank(pct=True)
        yrank = group['mean_oxygen'].rank(pct=True)
        ranks.append(pd.DataFrame({'x': xrank - xrank.mean(), 'y': yrank - yrank.mean()}))
    if not ranks:
        return 0, 0, np.nan
    values = pd.concat(ranks, ignore_index=True)
    return len(values), len(ranks), values['x'].corr(values['y']) if len(values) >= MIN_CORRELATION_RUNS else np.nan

correlation_rows = []
system_rows = []
for col in parameter_columns:
    n, rho = rho_pair(process_runs, col, 'mean_oxygen', MIN_CORRELATION_RUNS)
    within_n, within_spaces, within_rho = within_space_rho(process_runs, col)
    correlation_rows.append({'parameter': col, 'n_runs': n, 'pooled_spearman': rho,
                             'within_space_runs': within_n, 'within_space_count': within_spaces,
                             'within_space_rank_correlation': within_rho})
    for space, group in process_runs.groupby('space'):
        n_system, rho_system = rho_pair(group, col, 'mean_oxygen', MIN_SYSTEM_RUNS)
        if n_system >= MIN_SYSTEM_RUNS:
            system_rows.append({'space': space, 'parameter': col,
                                'n_runs': n_system, 'spearman': rho_system})
oxygen_process_correlations = pd.DataFrame(correlation_rows)
system_process_correlations = pd.DataFrame(system_rows, columns=['space', 'parameter', 'n_runs', 'spearman'])
print('Oxygen correlations: pooled and material-centered rank sensitivity:')
display(oxygen_process_correlations.sort_values('pooled_spearman', key=lambda s: s.abs(), ascending=False))
print('Within individual material spaces:')
display(system_process_correlations.sort_values(['space', 'parameter']))

def bootstrap_within_space_rank(frame, column, n_bootstraps=1000, seed=42):
    groups = []
    for _, group in frame[['space', column, 'mean_oxygen']].replace([np.inf, -np.inf], np.nan).dropna().groupby('space'):
        if len(group) >= 3 and group[column].nunique() > 1 and group['mean_oxygen'].nunique() > 1:
            groups.append(group[[column, 'mean_oxygen']].reset_index(drop=True))
    if len(groups) < 3:
        return None
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_bootstraps):
        pieces = []
        for group in groups:
            sampled = group.iloc[rng.integers(0, len(group), size=len(group))]
            xr = sampled[column].rank(pct=True)
            yr = sampled['mean_oxygen'].rank(pct=True)
            pieces.append(pd.DataFrame({'x': xr - xr.mean(), 'y': yr - yr.mean()}))
        ranks = pd.concat(pieces, ignore_index=True)
        estimate = ranks['x'].corr(ranks['y'])
        if np.isfinite(estimate):
            values.append(estimate)
    if len(values) < 0.9 * n_bootstraps:
        return None
    return np.percentile(values, [2.5, 97.5]), np.mean(np.asarray(values) >= 0)

bootstrap_candidates = oxygen_process_correlations[
    oxygen_process_correlations['within_space_count'].ge(3)
    & oxygen_process_correlations['within_space_rank_correlation'].notna()
    & oxygen_process_correlations['n_runs'].ge(20)
].sort_values('within_space_rank_correlation', key=lambda s: s.abs(), ascending=False).head(12)
bootstrap_rows = []
for parameter in bootstrap_candidates['parameter']:
    result = bootstrap_within_space_rank(process_runs, parameter)
    if result is not None:
        interval, positive_fraction = result
        summary = oxygen_process_correlations.set_index('parameter').loc[parameter]
        bootstrap_rows.append({'parameter': parameter, 'runs': summary['within_space_runs'],
                               'material_spaces': summary['within_space_count'],
                               'within_space_rho': summary['within_space_rank_correlation'],
                               'bootstrap_2_5%': interval[0], 'bootstrap_97_5%': interval[1],
                               'bootstrap_fraction_nonnegative': positive_fraction})
within_space_bootstrap = pd.DataFrame(bootstrap_rows)
print('Top material-centered correlations with stratified run-bootstrap intervals (descriptive; not multiplicity-corrected):')
display(within_space_bootstrap)
if not within_space_bootstrap.empty:
    ranked = within_space_bootstrap.sort_values('within_space_rho')
    correlation_figure = go.Figure(go.Scatter(
        x=ranked['within_space_rho'], y=ranked['parameter'], mode='markers',
        error_x={'type': 'data', 'symmetric': False,
                 'array': ranked['bootstrap_97_5%'] - ranked['within_space_rho'],
                 'arrayminus': ranked['within_space_rho'] - ranked['bootstrap_2_5%']},
        customdata=ranked[['runs', 'material_spaces', 'bootstrap_fraction_nonnegative']].to_numpy(),
        hovertemplate='%{y}<br>within-space rank rho: %{x:.3f}'
                      '<br>runs: %{customdata[0]}<br>spaces: %{customdata[1]}'
                      '<br>bootstrap fraction nonnegative: %{customdata[2]:.2f}<extra></extra>'))
    correlation_figure.add_vline(x=0, line_dash='dash', line_color='grey')
    correlation_figure.update_layout(template='plotly_white',
        title='Leading oxygen associations after material-space rank-centering')
    correlation_figure.update_xaxes(title='Material-centered Spearman-like rank correlation',
                                     showline=True, mirror=True, linecolor='black')
    correlation_figure.update_yaxes(title='Recorded process parameter',
                                     showline=True, mirror=True, linecolor='black')
    correlation_figure.show(config=PLOTLY_CONFIG)

valid_columns = [c for c in numeric if process_runs[c].notna().sum() >= MIN_CORRELATION_RUNS
                 and process_runs[c].nunique() >= 2]
if len(valid_columns) >= 2:
    corr_matrix = process_runs[valid_columns].corr(method='spearman', min_periods=MIN_CORRELATION_RUNS)
    present = process_runs[valid_columns].notna().astype(int)
    pair_counts = present.T.dot(present)
    heatmap = go.Figure(data=go.Heatmap(
        z=corr_matrix.to_numpy(), x=valid_columns, y=valid_columns, zmin=-1, zmax=1,
        colorscale='RdBu', reversescale=True, customdata=pair_counts.to_numpy(),
        hovertemplate='%{x} vs %{y}<br>Spearman rho: %{z:.3f}<br>paired runs: %{customdata}<extra></extra>'))
    heatmap.update_layout(template='plotly_white', title='Run-level Spearman correlations (pairwise coverage)')
    heatmap.update_xaxes(showline=True, mirror=True, linecolor='black', tickangle=45)
    heatmap.update_yaxes(showline=True, mirror=True, linecolor='black', autorange='reversed')
    heatmap.show(config=PLOTLY_CONFIG)
else:
    print('Insufficient variable parameters for a correlation matrix.')

rtp_run_flags = points.groupby('run_id')['rtp_input_any_time'].max().rename('ever_rtp_input')
process_runs = process_runs.merge(rtp_run_flags, on='run_id', validate='one_to_one')
non_rtp = process_runs[~process_runs['ever_rtp_input']]
print('Sensitivity excluding every run linked to a library ever used as RTP input:', len(non_rtp), 'runs')
rtp_sensitivity = pd.DataFrame([
    {'parameter': col, 'n_runs': rho_pair(non_rtp, col, 'mean_oxygen', MIN_CORRELATION_RUNS)[0],
     'spearman': rho_pair(non_rtp, col, 'mean_oxygen', MIN_CORRELATION_RUNS)[1]}
    for col in parameter_columns])
display(rtp_sensitivity)


## Exploratory separation of growth and thickness-dependent oxygen

For one material space at a time, fit the run-level descriptive model

$$x_{\mathrm O} = x_0 + \frac{A}{r} + \frac{C}{d}, \qquad d = rt.$$

Here $x_{\mathrm O}$ is the mean EDX oxygen atomic fraction; $r$ is the run-mean EDX-derived thickness divided by its deposition time in nm/min; $t$ is in minutes; and $d$ is the run-mean EDX thickness in nm. Under a simplified constant incorporated oxygen flux and fixed atomic density, $A$ (nm/min) is the incorporated oxygen flux divided by film atomic density. Under a simplified fixed-areal oxygen or measurement-bias model, $C$ (nm) is an effective oxygen-equivalent thickness. Neither coefficient alone is a direct measurement of flux or SiO2 thickness, and a fitted coefficient compatible with zero does **not** prove the corresponding effect is absent. $x_0$ is an empirical intercept, not necessarily a pure baseline oxygen concentration.

Each run receives equal weight: average points within each EDX map, maps within each library, then libraries within each run. This model uses the inverse of run-mean thickness/rate, not the mean of pointwise inverses. At a single deposition time $1/r$ and $1/d$ are proportional, so $A$ and $C$ cannot be identified separately. The fit checks for singularity and reports the correlation and condition number of its predictors. Intervals bootstrap whole runs, not EDX points. Negative coefficients are allowed because the fit is unconstrained and should not be read as negative physical fluxes.

**Caution:** a fixed areal contribution can reflect a real interface or surface layer as well as a substrate-fitting artifact. EDX oxygen and thickness are jointly fitted; the assumed density, film composition, substrate, film exposure, growth conditions and data timestamps may vary between runs. Base pressure is not oxygen partial pressure. Pressure/delay adjustment below is a sensitivity analysis, not a causal correction. The model and confidence intervals are exploratory and apply only over observed rates and thicknesses.

In [ ]:
MODEL_MATERIAL_SPACE = 'Cu-P-S'  # exact sputtering material_space; change to inspect another system
MODEL_BOOTSTRAPS = 5000
MODEL_RANDOM_SEED = 42

model_context = points[['run_id', 'seconds']].drop_duplicates()
if model_context['run_id'].duplicated().any():
    raise ValueError('A run has conflicting deposition times; inspect the source records.')
model_runs = run_means.merge(model_context, on='run_id', validate='one_to_one')
model_runs['time_min'] = model_runs['seconds'] / 60
model_runs['thickness_nm'] = model_runs['mean_rate_nm_min'] * model_runs['time_min']

# Parse timestamps separately to support mixed timezone/precision formats in older pandas versions.
map_times = points[['run_id', 'edx_id', 'edx_time', 'run_time']].drop_duplicates(['run_id', 'edx_id']).copy()
map_times['edx_datetime'] = map_times['edx_time'].map(lambda x: pd.to_datetime(x, utc=True, errors='coerce'))
map_times['run_datetime'] = map_times['run_time'].map(lambda x: pd.to_datetime(x, utc=True, errors='coerce'))
map_times['elapsed_days'] = (map_times['edx_datetime'] - map_times['run_datetime']).dt.total_seconds() / 86400
delays = map_times.groupby('run_id', as_index=False)['elapsed_days'].median()
model_runs = model_runs.merge(delays, on='run_id', how='left', validate='one_to_one')
model_runs = model_runs[model_runs['space'] == MODEL_MATERIAL_SPACE].copy()
model_runs = model_runs[
    np.isfinite(model_runs['mean_oxygen']) & np.isfinite(model_runs['mean_rate_nm_min'])
    & np.isfinite(model_runs['thickness_nm']) & (model_runs['mean_rate_nm_min'] > 0)
    & (model_runs['thickness_nm'] > 0)
].copy()
model_runs['inverse_rate'] = 1 / model_runs['mean_rate_nm_min']
model_runs['inverse_thickness'] = 1 / model_runs['thickness_nm']

def checked_design(frame, predictors):
    if len(frame) < len(predictors) + 4:
        raise ValueError(f'Only {len(frame)} runs for {len(predictors)} predictors; need at least {len(predictors) + 4}.')
    values = frame[predictors].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('Model predictors contain missing or non-finite values.')
    scales = values.std(axis=0)
    if np.any(scales == 0):
        raise ValueError('A predictor has no variation; A and C may not be identifiable.')
    standardized = (values - values.mean(axis=0)) / scales
    condition = np.linalg.cond(standardized)
    design = np.column_stack([np.ones(len(frame)), values])
    if np.linalg.matrix_rank(design) < design.shape[1] or condition > 100:
        raise ValueError(f'Model is singular or ill-conditioned (standardized condition number {condition:.2f}).')
    return design, condition

def fit_with_run_bootstrap(frame, predictors, n_bootstraps=MODEL_BOOTSTRAPS):
    design, condition = checked_design(frame, predictors)
    y = frame['mean_oxygen'].to_numpy(dtype=float)
    coef = np.linalg.lstsq(design, y, rcond=None)[0]
    rng = np.random.default_rng(MODEL_RANDOM_SEED)
    boot = []
    for _ in range(n_bootstraps):
        index = rng.integers(0, len(frame), len(frame))
        sampled = design[index]
        if np.linalg.matrix_rank(sampled) == sampled.shape[1]:
            boot.append(np.linalg.lstsq(sampled, y[index], rcond=None)[0])
    if len(boot) < 0.9 * n_bootstraps:
        raise ValueError(f'Only {len(boot)} of {n_bootstraps} bootstrap fits were identifiable; intervals are unreliable.')
    lower, upper = np.quantile(np.asarray(boot), [0.025, 0.975], axis=0)
    names = ['intercept'] + predictors
    result = pd.DataFrame({'term': names, 'estimate': coef, 'bootstrap_2_5%': lower,
                           'bootstrap_97_5%': upper})
    return result, condition, len(boot)

predictors = ['inverse_rate', 'inverse_thickness']
coefficients, condition, n_boot = fit_with_run_bootstrap(model_runs, predictors)
coefficients['meaning'] = ['x0 (atomic fraction)', 'A (nm/min)', 'C (nm)']
print(f'{MODEL_MATERIAL_SPACE}: {len(model_runs)} runs; {n_boot}/{MODEL_BOOTSTRAPS} identifiable run bootstraps.')
print(f"Deposition times: {model_runs['time_min'].min():.2f}–{model_runs['time_min'].max():.2f} min; "
      f"predictor correlation: {model_runs['inverse_rate'].corr(model_runs['inverse_thickness']):.3f}; "
      f'standardized condition number: {condition:.2f}.')
display(coefficients)
display(model_runs[['time_min', 'mean_rate_nm_min', 'thickness_nm', 'mean_oxygen',
                    'pressure_torr', 'elapsed_days']].describe())

In [ ]:
def leave_one_run_out_rmse(frame, predictors):
    y = frame['mean_oxygen'].to_numpy(dtype=float)
    values = frame[predictors].to_numpy(dtype=float)
    design = np.column_stack([np.ones(len(frame)), values])
    predictions = []
    for index in range(len(frame)):
        keep = np.arange(len(frame)) != index
        if np.linalg.matrix_rank(design[keep]) < design.shape[1]:
            return np.nan
        coef = np.linalg.lstsq(design[keep], y[keep], rcond=None)[0]
        predictions.append(design[index] @ coef)
    return float(np.sqrt(np.mean((np.asarray(predictions) - y) ** 2)))

comparison = pd.DataFrame([
    {'model': label, 'leave_one_run_out_rmse': leave_one_run_out_rmse(model_runs, cols)}
    for label, cols in [('constant', []), ('growth proxy: A/r', ['inverse_rate']),
                        ('thickness proxy: C/d', ['inverse_thickness']),
                        ('both proxies', predictors)]
])
print('Run-held-out prediction error (oxygen atomic fraction; lower is better):')
display(comparison)
if comparison['leave_one_run_out_rmse'].isna().any():
    print('A held-out training fold was rank-deficient; that model has no valid RMSE.')

adjusted = model_runs[(model_runs['pressure_torr'] > 0) & (model_runs['elapsed_days'] > 0)].copy()
adjusted['log10_pressure_torr'] = np.log10(adjusted['pressure_torr'])
adjusted['log10_elapsed_days'] = np.log10(adjusted['elapsed_days'])
adjusted_predictors = predictors + ['log10_pressure_torr', 'log10_elapsed_days']
print(f'Pressure and recorded EDX-delay sensitivity: {len(adjusted)} of {len(model_runs)} runs have positive values.')
print('Elapsed time is based on recorded archive timestamps, which may not equal measurement or exposure time.')
try:
    adjusted_coefficients, adjusted_condition, adjusted_boot = fit_with_run_bootstrap(adjusted, adjusted_predictors)
except ValueError as exc:
    print(f'Adjusted model unavailable: {exc}')
else:
    print(f'Adjusted standardized condition number: {adjusted_condition:.2f}; identifiable bootstraps: {adjusted_boot}.')
    display(adjusted_coefficients)

model_runs['fitted_oxygen'] = (coefficients.loc[0, 'estimate']
    + coefficients.loc[1, 'estimate'] * model_runs['inverse_rate']
    + coefficients.loc[2, 'estimate'] * model_runs['inverse_thickness'])
model_figure = px.scatter(
    model_runs, x='mean_oxygen', y='fitted_oxygen', color='time_min',
    hover_data=['run_lab_id', 'run_id', 'mean_rate_nm_min', 'thickness_nm',
                'pressure_torr', 'elapsed_days', 'n_libraries'], template='plotly_white',
    labels={'mean_oxygen': 'Observed mean oxygen atomic fraction',
            'fitted_oxygen': 'In-sample fitted oxygen atomic fraction', 'time_min': 'Deposition time (min)'},
    title=f'{MODEL_MATERIAL_SPACE}: observed versus fitted run-mean oxygen (not held-out)',
)
low = min(model_runs['mean_oxygen'].min(), model_runs['fitted_oxygen'].min())
high = max(model_runs['mean_oxygen'].max(), model_runs['fitted_oxygen'].max())
model_figure.add_shape(type='line', x0=low, x1=high, y0=low, y1=high, line={'color': 'black', 'dash': 'dash'})
model_figure.update_xaxes(showline=True, mirror=True, linecolor='black')
model_figure.update_yaxes(showline=True, mirror=True, linecolor='black')
model_figure.show(config=PLOTLY_CONFIG)

## All eligible sputtering runs: pooled and material-specific sensitivity

The pooled model fits a different intercept for each recorded material space, but assumes a **common** $A$ and $C$ across material spaces. This is a working statistical assumption, not a physical conclusion: oxygen sticking, density, oxide chemistry and EDX response may vary with composition and substrate. Each run has equal weight; points or libraries sharing a run are not independent replicates. Runs in a material space represented only once are included in coverage but provide no within-space slope information. Runs without a recorded material space are excluded because they cannot be assigned a meaningful material baseline.

The stratified bootstrap resamples runs **within** each material space, preserving the number of runs per space. It quantifies conditional statistical uncertainty, not uncertainty from violations of the shared-coefficient assumption, observational confounding or LayerProbe errors. Leave-one-material-space-out fits and separate fits for every material space show sensitivity to this assumption. The separate fit requires at least six runs and a sufficiently identifiable design; otherwise the coverage table gives the reason it is unavailable. Fits with only six or seven runs are particularly unstable, even when the algorithm returns an interval. More pooled runs do not necessarily narrow an interval or establish an oxygen mechanism.

In [ ]:
POOLED_BOOTSTRAPS = 3000
all_model_runs = run_means.merge(model_context, on='run_id', validate='one_to_one')
all_model_runs['time_min'] = all_model_runs['seconds'] / 60
all_model_runs['thickness_nm'] = all_model_runs['mean_rate_nm_min'] * all_model_runs['time_min']
valid = (np.isfinite(all_model_runs['mean_oxygen']) & np.isfinite(all_model_runs['mean_rate_nm_min'])
         & np.isfinite(all_model_runs['thickness_nm']) & (all_model_runs['mean_rate_nm_min'] > 0)
         & (all_model_runs['thickness_nm'] > 0) & all_model_runs['space'].notna()
         & (all_model_runs['space'] != 'Unknown'))
print(f'Pooled coverage: {valid.sum()} of {len(all_model_runs)} eligible runs have a known material space and valid rate/thickness.')
all_model_runs = all_model_runs[valid].copy()
all_model_runs['inverse_rate'] = 1 / all_model_runs['mean_rate_nm_min']
all_model_runs['inverse_thickness'] = 1 / all_model_runs['thickness_nm']
group_sizes = all_model_runs.groupby('space').size()
print(f'Material spaces: {len(group_sizes)}; singleton spaces: {(group_sizes == 1).sum()}; '
      f'runs in singleton spaces: {group_sizes[group_sizes == 1].sum()}.')

def pooled_arrays(frame, predictors):
    return [group[predictors + ['mean_oxygen']].to_numpy(dtype=float)
            for _, group in frame.groupby('space', sort=True)]

def pooled_solve(groups):
    centered = np.concatenate([group - group.mean(axis=0) for group in groups], axis=0)
    x, y = centered[:, :-1], centered[:, -1]
    scales = x.std(axis=0)
    if np.any(scales == 0) or np.linalg.matrix_rank(x) < x.shape[1]:
        raise ValueError('Within-material predictors are singular or have no variation.')
    condition = np.linalg.cond(x / scales)
    if condition > 100:
        raise ValueError(f'Within-material predictors are ill-conditioned ({condition:.2f}).')
    return np.linalg.lstsq(x, y, rcond=None)[0], condition

groups = pooled_arrays(all_model_runs, predictors)
pooled_coef, pooled_condition = pooled_solve(groups)
pooled_rng = np.random.default_rng(MODEL_RANDOM_SEED)
pooled_bootstrap = []
for _ in range(POOLED_BOOTSTRAPS):
    sampled = [group[pooled_rng.integers(0, len(group), len(group))] for group in groups]
    try:
        boot_coef, _ = pooled_solve(sampled)
    except ValueError:
        continue
    pooled_bootstrap.append(boot_coef)
if len(pooled_bootstrap) < 0.9 * POOLED_BOOTSTRAPS:
    raise ValueError(f'Only {len(pooled_bootstrap)}/{POOLED_BOOTSTRAPS} pooled bootstrap fits were identifiable.')
pooled_low, pooled_high = np.quantile(np.asarray(pooled_bootstrap), [0.025, 0.975], axis=0)
pooled_coefficients = pd.DataFrame({'term': ['A (nm/min)', 'C (nm)'], 'estimate': pooled_coef,
                                    'bootstrap_2_5%': pooled_low, 'bootstrap_97_5%': pooled_high})
print(f'Pooled fixed-material-intercept fit; standardized condition number {pooled_condition:.2f}; '
      f'{len(pooled_bootstrap)}/{POOLED_BOOTSTRAPS} identifiable stratified run bootstraps.')
display(pooled_coefficients)

leave_space_rows = []
for space in group_sizes.index:
    subset = all_model_runs[all_model_runs['space'] != space]
    try:
        fit, condition = pooled_solve(pooled_arrays(subset, predictors))
    except ValueError as exc:
        print(f'Excluding {space}: model unavailable: {exc}')
        continue
    leave_space_rows.append({'excluded_space': space, 'excluded_runs': int(group_sizes[space]),
                             'remaining_runs': len(subset), 'A_nm_min': fit[0], 'C_nm': fit[1]})
leave_space_out = pd.DataFrame(leave_space_rows)
print('Leave-one-material-space-out sensitivity:')
display(leave_space_out.sort_values('excluded_runs', ascending=False))

system_rows, system_coverage = [], []
for space, group in all_model_runs.groupby('space'):
    try:
        result, condition, n_boot = fit_with_run_bootstrap(group, predictors, n_bootstraps=2000)
    except ValueError as exc:
        system_coverage.append({'space': space, 'runs': len(group), 'status': f'Not fitted: {exc}'})
        continue
    system_coverage.append({'space': space, 'runs': len(group), 'status': 'Fitted' if len(group) >= 8 else 'Fitted; very few runs'})
    system_rows.append({'space': space, 'runs': len(group), 'condition': condition,
                        'identifiable_bootstraps': n_boot, 'x0': result.loc[0, 'estimate'],
                        'A_nm_min': result.loc[1, 'estimate'], 'A_low': result.loc[1, 'bootstrap_2_5%'],
                        'A_high': result.loc[1, 'bootstrap_97_5%'], 'C_nm': result.loc[2, 'estimate'],
                        'C_low': result.loc[2, 'bootstrap_2_5%'], 'C_high': result.loc[2, 'bootstrap_97_5%']})
system_fits = pd.DataFrame(system_rows)
print('Separate fit coverage for every material space:')
display(pd.DataFrame(system_coverage))
print('Separate material-space fits (run-bootstrap intervals may be very wide):')
display(system_fits)

## Compare measurements and fits for each material system

Select a material space in the dropdown. Blue circles are measured mean oxygen for each run; orange crosses are oxygen fitted **for the same run**, at that run's rate, thickness and deposition time. The grey vertical line joins each observed/fitted pair. The left panel shows oxygen against rate and the right panel against thickness. The fit depends on both rate and thickness, so fitted points need not follow a single curve in either panel. These are in-sample fits, not held-out predictions; use the run-held-out error table above for the selected single-system fit. The earlier observed-versus-fitted diagnostic intentionally has oxygen on both axes: proximity to its diagonal indicates how closely the model reproduces each measurement.

In [ ]:
from plotly.subplots import make_subplots

if system_fits.empty:
    print('No chemical system has enough identifiable runs for an individual fit plot.')
else:
    individual_figure = make_subplots(rows=1, cols=2, subplot_titles=('Oxygen versus deposition rate', 'Oxygen versus thickness'))
    spaces = system_fits['space'].tolist()
    if MODEL_MATERIAL_SPACE in spaces:
        spaces.remove(MODEL_MATERIAL_SPACE)
        spaces.insert(0, MODEL_MATERIAL_SPACE)
    for space_index, space in enumerate(spaces):
        fit = system_fits.loc[system_fits['space'] == space].iloc[0]
        group = all_model_runs[all_model_runs['space'] == space].copy()
        group['fitted_oxygen'] = fit['x0'] + fit['A_nm_min'] * group['inverse_rate'] + fit['C_nm'] * group['inverse_thickness']
        visible = space_index == 0
        custom = group[['run_lab_id', 'run_id', 'time_min', 'mean_rate_nm_min',
                        'thickness_nm', 'mean_oxygen', 'fitted_oxygen']].to_numpy()
        hover = ('Run: %{customdata[0]}<br>ID: %{customdata[1]}<br>Time: %{customdata[2]:.1f} min'
                 '<br>Rate: %{customdata[3]:.2f} nm/min<br>Thickness: %{customdata[4]:.1f} nm'
                 '<br>Measured O: %{customdata[5]:.4f}<br>Fitted O: %{customdata[6]:.4f}<extra></extra>')
        for col, x_column in [(1, 'mean_rate_nm_min'), (2, 'thickness_nm')]:
            line_x, line_y = [], []
            for x, observed, predicted in zip(group[x_column], group['mean_oxygen'], group['fitted_oxygen']):
                line_x.extend([x, x, None])
                line_y.extend([observed, predicted, None])
            individual_figure.add_trace(go.Scatter(x=line_x, y=line_y, mode='lines',
                line={'color': 'lightgrey', 'width': 1}, hoverinfo='skip', showlegend=False, visible=visible), row=1, col=col)
            for column, label, symbol, color in [('mean_oxygen', 'Measured oxygen', 'circle', '#1f77b4'),
                                                  ('fitted_oxygen', 'Fitted oxygen', 'x', '#d95f02')]:
                individual_figure.add_trace(go.Scatter(
                    x=group[x_column], y=group[column], mode='markers', name=label, legendgroup=label,
                    showlegend=(col == 1), visible=visible, customdata=custom, hovertemplate=hover,
                    marker={'symbol': symbol, 'color': color, 'size': 8}), row=1, col=col)
    buttons = []
    for index, space in enumerate(spaces):
        visible = [j // 6 == index for j in range(len(individual_figure.data))]
        buttons.append({'label': space, 'method': 'update', 'args': [
            {'visible': visible}, {'title': f'{space}: measured and fitted oxygen (one point per run)'}]})
    individual_figure.update_layout(template='plotly_white',
        title=f'{spaces[0]}: measured and fitted oxygen (one point per run)',
        updatemenus=[{'buttons': buttons, 'x': 0, 'y': 1.18, 'xanchor': 'left'}],
        legend={'orientation': 'h', 'y': -0.2})
    individual_figure.update_xaxes(title_text='EDX-derived deposition rate (nm/min)', type='log',
                                   showline=True, mirror=True, linecolor='black', row=1, col=1)
    individual_figure.update_xaxes(title_text='EDX-derived thickness (nm)', type='log',
                                   showline=True, mirror=True, linecolor='black', row=1, col=2)
    individual_figure.update_yaxes(title_text='Oxygen atomic fraction', showline=True, mirror=True,
                                   linecolor='black', row=1, col=1)
    individual_figure.update_yaxes(title_text='Oxygen atomic fraction', showline=True, mirror=True,
                                   linecolor='black', row=1, col=2)
    individual_figure.show(config=PLOTLY_CONFIG)